# Gradio lab: a web interface for the support chatbot

**Problem.** The food-delivery support assistant from Module 3 lives in a notebook. Only someone who can run Python can try it. Before the company builds anything bigger (Units 4.1–4.2), the support team wants to **use** it: type questions, ask follow-ups, and tell the engineers what works and what does not.

**Gradio** is a Python library that turns a function into a web app in a few lines. You will use it to put the assistant behind a chat interface, share it with a classmate through a link, and then **test it**: first automatically, for correctness and speed, and then with a real person, for user experience.

**Learning objectives**

1. Explain Gradio's model: a Python function, input and output **components**, and `launch()`.
2. Build a `gr.Interface` for a plain function and for a language model with decoding controls.
3. Build a chatbot with `gr.ChatInterface`, passing the conversation history to the model correctly.
4. Test the chatbot's answers and latency automatically, without the interface.
5. Run a short user test with a partner, and turn what you observe into improvements.
6. Explain what a public share link exposes, and why a Gradio prototype is not a production system.

**Prerequisites:** [RAG with speech](../../../module3/lectures/rag/rag_speech.ipynb) (the assistant, its policies and its failures) · [From model to application](../../lectures/applications_design/applications_and_design.md) (Units 4.1–4.2: requirements, guardrails, evaluation plans) · chat templates and greedy vs. sampled decoding (Modules 2–3).

**Data and model**

- **Policies:** the 13 short help-centre articles from the Module 3 RAG lecture. They are **synthetic**, invented for the course, so no real company's policy is involved.
- **Model:** [Qwen/Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) on a GPU (about 3.1 GB download), or [Qwen/Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) on a CPU (about 1 GB). Both are Apache-2.0. The smaller model is slower on a CPU and follows instructions noticeably worse.
- **No retrieval.** All 13 articles together are short enough to fit in the prompt, so this lab puts all of them there and skips retrieval. The lab is about the interface. For a larger document collection, the retriever from Module 3 would sit at the same place in the code.

**Environment and compute**

- **Kaggle notebook, GPU T4, internet on.** Internet is needed for the downloads **and** for the share link. On a CPU-only machine everything runs, but each answer takes noticeably longer.
- Packages: `gradio` (install cell below), plus `transformers` and `torch` (preinstalled on Kaggle). The code works with Gradio 5 and Gradio 6; the one difference that matters is handled in Part 3.
- No training. Expect the model download to take a minute or two, and each answer to take a few seconds on a T4. Plan about **two hours** in total, including the partner test.

**Files.** The chat log from Part 5 is written to `/kaggle/working/gradio_lab/chat_log.csv` (or `./gradio_lab_outputs/chat_log.csv` elsewhere). Rerunning Part 5 adds rows to it; delete the file to start again.

**How to work:** run the cells in order. The exercise implementations below are complete; run their provided checks and inspect the measured results. Each app you launch prints a public link ending in `.gradio.live`: open it in a new browser tab. Launching a new app closes the previous one.

**Marks (20):** Ex. 1 — 2 · Ex. 2 — 3 · Ex. 3 — 4 · Ex. 4 — 4 · Ex. 5 — 5 · reflection — 2.

**Submission: Jiwak.** Exercises 1–5 contain completed implementations and saved GPU execution evidence using Qwen2.5-1.5B-Instruct, float16, on an NVIDIA GeForce RTX 3050 6GB Laptop GPU. The supplied implementation checks passed in that recorded run. On 24 November 2026, Codex completed an independent AI review of the saved task responses, with policy judgments, reviewer ratings, ranked problems and the allergy-fix comparison. This review is not a human participant session; acceptance as a substitute for the partner exercise is the instructor's decision.

The preserved run used PyTorch 2.11.0+cu130, Transformers 5.5.4 and Gradio 6.31.0. This notebook requires CUDA. Local apps use `share=False`; set `GRADIO_LAB_SHARE=1` for a public test link. Pause before cleanup to use an app. Model results below are preserved historical measurements, not a new GPU run.

## Setup

**Installation (Kaggle).** Run the next cell once per session; it takes about a minute. If `gradio` was already imported in this session, restart the kernel afterwards and continue from the cell after it.

In [1]:
%pip install -q -U gradio

Local run: dependencies installed in /tmp/gradio_assignment_deps. GPU execution; package versions below.


In [2]:
import os
import csv
import re
import statistics
import time
from datetime import datetime
from pathlib import Path
from threading import Thread

import gradio as gr
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer

torch.manual_seed(0)
if not torch.cuda.is_available():
    raise RuntimeError("This submission requires a GPU. Enable a GPU accelerator in Kaggle/Colab, or run with NVIDIA driver access locally.")
DEVICE = "cuda"
GRADIO_MAJOR = int(gr.__version__.split(".")[0])
OUT = Path(os.environ.get("GRADIO_LAB_OUT", "/kaggle/working/gradio_lab" if Path("/kaggle/working").is_dir() else "gradio_lab_outputs"))
OUT.mkdir(parents=True, exist_ok=True)
SHARE = os.environ.get("GRADIO_LAB_SHARE", "1" if Path("/kaggle/working").is_dir() else "0") == "1"
# Use GRADIO_LAB_SHARE=1 for an intentional public partner-test link.

print("gradio", gr.__version__, "| transformers", transformers.__version__, "| torch", torch.__version__)
print("device:", DEVICE, "| outputs:", OUT)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


gradio 6.31.0 | transformers 5.5.4 | torch 2.11.0+cu130
device: cuda | outputs: /home/jiwak/GenAi_Notes/Gen_AI_Lab/generative_ai_shared/module3/labs/gradio_chatbot/gradio_lab_outputs
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


**Why a share link?** Your notebook runs on a Kaggle machine, not on your laptop. Gradio starts a small web server on that machine, which your browser cannot reach directly. With `share=True`, Gradio opens a tunnel through its own servers and prints a public address ending in `.gradio.live`. Gradio's documentation states that share links expire after one week; on Kaggle the link also stops working when your session ends. **Anyone who has the link can use your app**, and your GPU, until then. Part 6 returns to what this means.

The helper below closes the previously launched app before starting a new one, so old servers and links do not pile up.

In [3]:
_running = []


def launch(demo):
    """Close the previously launched app, then launch `demo` with a share link."""
    while _running:
        try:
            _running.pop().close()
        except Exception:
            pass
    demo.launch(share=SHARE, prevent_thread_lock=True)
    _running.append(demo)

## Part 1 — Gradio in ten minutes

A Gradio app has three parts:

```text
   input components ──values──►  your Python function  ──return value──►  output components
   (textbox, slider, radio)       fn(a, b, ...)                            (textbox, label, image)
                         └──────────────── launch(): a web page that calls fn on every submit ─────────────┘
```

1. **A function.** Ordinary Python. Gradio knows nothing about models: the function can apply a rule, call a model, or query a database.
2. **Components.** Each input component supplies one argument to the function, **in order**. Each output component displays one return value. Commonly used components:

| Component | Shows | Value passed to / from the function |
| --- | --- | --- |
| `gr.Textbox` | A text field | `str` |
| `gr.Number` | A number field | `float` (or `int` with `precision=0`) |
| `gr.Slider` | A slider with `minimum`, `maximum`, `step` | `float` |
| `gr.Radio`, `gr.Dropdown` | A fixed list of `choices` | the chosen `str` |
| `gr.Checkbox` | A tick box | `bool` |
| `gr.Image`, `gr.Audio` | Media upload, microphone, or display | a NumPy array or a file path |

3. **`launch()`.** Starts a web server. In a notebook it returns immediately, and the app keeps running in the background until it is closed.

`gr.Interface` builds a complete page (inputs, a Submit button, outputs, optional examples) from just these pieces. Here it wraps the **late-delivery rule** from the policies. This is a rule, not a model: as Unit 4.1 argued, an exact policy value should come from a rule, never from text generation.

In [4]:
def late_delivery_coupon(minutes_late, cause):
    """The late-delivery policy as a rule (synthetic policy from the Module 3 RAG lecture)."""
    if cause != "None of these":
        return f"No coupon: delays caused by {cause.lower()} do not qualify."
    if minutes_late > 30:
        return "You get a Rs 50 coupon, added automatically within 24 hours and valid for 14 days."
    return f"No coupon: the order must be more than 30 minutes late (this one was {minutes_late:.0f} minutes late)."


demo_rule = gr.Interface(
    fn=late_delivery_coupon,
    inputs=[gr.Slider(0, 120, value=40, step=1, label="Minutes after the promised time"),
            gr.Radio(["None of these", "Heavy rain", "An incorrect address", "A customer who cannot be reached"],
                     value="None of these", label="Cause of the delay")],
    outputs=gr.Textbox(label="Compensation"),
    title="Late-delivery coupon checker",
    description="Rule-based: this app contains no model.",
    examples=[[40, "None of these"], [45, "Heavy rain"], [20, "None of these"]],
    flagging_mode="never",   # hide the Flag button for now
)
launch(demo_rule)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Open the link, move the slider, pick a cause, and press **Submit**. Click an example row to fill the inputs. Each submit calls `late_delivery_coupon(minutes_late, cause)` on the Kaggle machine and shows its return value.

### Exercise 1 — An interface for the cancellation policy (2 marks)

The cancellation article says: *cancelling is free within 60 seconds of placing the order; after the restaurant accepts the order, cancelling costs 50 percent of the order value; once the rider has picked up the food, the order cannot be cancelled or refunded; if the restaurant cancels, you always get a full refund.*

1. Complete `cancellation_charge(order_value, stage)` so that it returns a sentence giving the charge in rupees, using the four `STAGES` below.
2. Build `demo_cancel`, a `gr.Interface` with a `gr.Number` for the order value, a `gr.Radio` (or `gr.Dropdown`) for the stage, a `gr.Textbox` output, a title, and at least three examples. Then launch it and try it.

In [5]:
import math
STAGES = ["Within 60 seconds of ordering", "After the restaurant accepted", "After the rider picked up", "The restaurant cancelled"]
def cancellation_charge(order_value, stage):
    try:
        value = float(order_value)
    except (TypeError, ValueError):
        return "Enter a valid non-negative order value in rupees."
    if not math.isfinite(value) or value < 0:
        return "Enter a valid non-negative order value in rupees."
    if stage == STAGES[0]:
        return "Cancellation is free within 60 seconds: charge Rs 0."
    if stage == STAGES[1]:
        return f"Cancellation costs 50 percent: charge Rs {value * 0.5:,.2f}."
    if stage == STAGES[2]:
        return f"The order cannot be cancelled or refunded after pickup; Rs {value:,.2f} remains payable."
    if stage == STAGES[3]:
        return f"Cancellation charge Rs 0; full refund of Rs {value:,.2f} because the restaurant cancelled."
    return "Select one of the four listed stages."

demo_cancel = gr.Interface(
    cancellation_charge,
    inputs=[gr.Number(value=400, minimum=0, label="Order value (Rs)"), gr.Radio(STAGES, value=STAGES[0], label="Order stage")],
    outputs=gr.Textbox(label="Cancellation policy"), title="Cancellation charge checker",
    description="Synthetic course policy. This checker explains charges; it cannot cancel an order.",
    examples=[[400, s] for s in STAGES], flagging_mode="never")
launch(demo_cancel)

Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
Live Gradio API cancellation check: Cancellation costs 50 percent: charge Rs 200.00.


In [6]:
# Check for Exercise 1
assert "200" in cancellation_charge(400, STAGES[1]), "50 percent of Rs 400 should be Rs 200"
assert "free" in cancellation_charge(400, STAGES[0]).lower(), "cancelling within 60 seconds is free"
assert isinstance(demo_cancel, gr.Interface)
for stage in STAGES:
    print(f"{stage:<32} -> {cancellation_charge(400, stage)}")
print("Exercise 1 checks passed.")

Within 60 seconds of ordering    -> Cancellation is free within 60 seconds: charge Rs 0.
After the restaurant accepted    -> Cancellation costs 50 percent: charge Rs 200.00.
After the rider picked up        -> The order cannot be cancelled or refunded after pickup; Rs 400.00 remains payable.
The restaurant cancelled         -> Cancellation charge Rs 0; full refund of Rs 400.00 because the restaurant cancelled.
Exercise 1 checks passed.


## Part 2 — A model behind the interface

Now the function calls a language model. Loading the model takes most of the time, so it is done **once**, outside the function. The function then only formats a prompt and generates.

> **Download:** the next cell downloads `Qwen2.5-1.5B-Instruct` (about 3.1 GB) on a GPU, or `Qwen2.5-0.5B-Instruct` (about 1 GB) on a CPU.

In [7]:
import os
CKPT = os.environ.get("GRADIO_LAB_MODEL", "Qwen/Qwen2.5-1.5B-Instruct" if DEVICE == "cuda" else "Qwen/Qwen2.5-0.5B-Instruct")
if DEVICE == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
tokenizer = AutoTokenizer.from_pretrained(CKPT)
model = AutoModelForCausalLM.from_pretrained(CKPT, dtype=torch.float16 if DEVICE == "cuda" else torch.float32)
model = model.to(DEVICE).eval()
print("model:", CKPT)


def _inputs(messages):
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    return tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(DEVICE)


def _decoding(temperature):
    # temperature 0 means greedy decoding: always the most likely token, so the same input gives the same answer
    return dict(do_sample=False) if temperature == 0 else dict(do_sample=True, temperature=temperature, top_p=0.95)


@torch.no_grad()
def chat(messages, max_new_tokens=200, temperature=0.0):
    """Generate one reply to a list of {"role", "content"} messages."""
    batch = _inputs(messages)
    out = model.generate(**batch, max_new_tokens=max_new_tokens, **_decoding(temperature))
    return tokenizer.decode(out[0, batch["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def stream_chat(messages, max_new_tokens=200, temperature=0.0):
    """Like chat(), but yields the reply so far after every new piece of text."""
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    kwargs = dict(**_inputs(messages), max_new_tokens=max_new_tokens, streamer=streamer, **_decoding(temperature))
    thread = Thread(target=torch.inference_mode()(model.generate), kwargs=kwargs)   # generate in the background ...
    thread.start()
    text = ""
    for piece in streamer:                                   # ... while we read its output as it arrives
        text += piece
        yield text
    thread.join()

Loading weights: 100%|##########| 338/338 [00:00<00:00, 721.75it/s]
model: /home/jiwak/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306


**The policies and the system prompt.** All 13 synthetic articles go into the system prompt, each labelled with its id so that the model can cite it. The prompt asks for short answers, citations, and a fixed refusal sentence when the policies do not answer the question. These are the same rules as in Module 3.

In [8]:
ARTICLES = {   # synthetic policies from the Module 3 RAG lecture, condensed
    "refunds": "Refunds to the app wallet arrive within 2 hours. Refunds to the original payment method take 5 to 7 working days for credit and debit cards, and 2 to 3 working days for UPI. If a refund has not arrived after 7 working days, contact support with the refund reference number shown on the order page.",
    "missing_items": "If an item is missing, report it from the order page within 48 hours of delivery, with a photo of the bag and the bill if you can. We refund the price of each missing item. The delivery fee is refunded only if the whole order is missing.",
    "late_delivery": "If your order arrives more than 30 minutes after the promised time, we add a Rs 50 coupon to your account automatically within 24 hours. The coupon is valid for 14 days. Delays caused by heavy rain, an incorrect address, or a customer who cannot be reached do not qualify.",
    "food_safety": "Report a food-safety problem within 24 hours of delivery from the order page, with a photo. We refund the full order value for every confirmed food-safety complaint. If you needed medical treatment, we reimburse medical expenses up to Rs 5,000; share the prescription and receipts within 7 days of the order.",
    "payment_failures": "If money was deducted but the order failed, the amount is reversed automatically within 48 hours; do not pay again. If you were charged twice for one order, the duplicate charge is reversed within 5 working days.",
    "cancellation": "Cancelling is free within 60 seconds of placing the order. After the restaurant accepts the order, cancelling costs 50 percent of the order value. Once the rider has picked up the food, the order cannot be cancelled or refunded. If the restaurant cancels, you always get a full refund.",
    "coupons": "Only one coupon can be applied to each order; coupons cannot be combined. Expired coupons cannot be reactivated, and coupons cannot be exchanged for cash.",
    "gold_membership": "Gold membership costs Rs 149 per month or Rs 1,199 per year. Members get free delivery on orders above Rs 199 from restaurants within 7 km, and priority phone support. Cancel any time; benefits continue until the end of the paid period, with no partial refunds.",
    "allergens": "Allergen information is provided by each restaurant. We cannot guarantee that any dish is free from nuts, gluten, dairy or other allergens. If you have a severe allergy, message the restaurant from the order page before ordering, and add a cooking note.",
    "address_change": "You can change the delivery address only before the rider picks up the order, and only to a new address within 2 km of the original one.",
    "tipping": "You can tip before checkout or up to 24 hours after delivery. The delivery partner receives 100 percent of the tip. Tips are never added automatically.",
    "account_deletion": "Delete your account from Settings, then Privacy, then Delete account. Your profile, addresses and order history are deleted within 30 days. Wallet balance is lost when the account is deleted.",
    "contact_support": "Chat support is available 24 hours a day from the Help section. Phone support is available from 8 am to 11 pm for Gold members only.",
}
REFUSAL = "I do not have that information. Please contact support."
SYSTEM = (
    "You are the support assistant for a food-delivery app. Answer using only the policies below. "
    "After each fact, cite the policy id in square brackets, for example [refunds]. "
    f'If the policies do not contain the answer, reply exactly: "{REFUSAL}" '
    "You cannot take actions such as issuing refunds or cancelling orders. Keep answers to two or three sentences.\n\n"
    "POLICIES\n" + "\n".join(f"[{k}] {v}" for k, v in ARTICLES.items())
)
print("system prompt:", len(tokenizer(SYSTEM)["input_ids"]), "tokens")

system prompt: 787 tokens


### Exercise 2 — A question-answering interface with decoding controls (3 marks)

1. Complete `answer(question, temperature, max_new_tokens)`. It builds a two-message list (the `SYSTEM` prompt, then the question as the `"user"` message) and returns `chat(...)`. Note that a slider passes a `float`: convert `max_new_tokens` with `int()`.
2. Build `demo_qa`: a `gr.Interface` with a `gr.Textbox` for the question, a temperature slider from 0 to 1.5 (default 0), a slider for `max_new_tokens` from 16 to 300 (default 150), and a `gr.Textbox` output. Add a few example questions. Launch it.

In [9]:
def answer(question, temperature, max_new_tokens):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    return chat(messages, temperature=float(temperature), max_new_tokens=int(max_new_tokens))

demo_qa = gr.Interface(
    answer,
    inputs=[gr.Textbox(label="Policy question"), gr.Slider(0, 1.5, value=0, step=0.1, label="Temperature (0 = greedy)"),
            gr.Slider(16, 300, value=150, step=1, label="Maximum new tokens")],
    outputs=gr.Textbox(label="Answer"), title="Food-delivery policy questions",
    description="Policy explanations only; no order access or refund actions. Answers may be wrong.",
    examples=[["How long does a refund to my UPI account take?", 0, 150],
              ["Can I use two coupons?", 0, 150], ["Refund my last order now.", 0, 150]], flagging_mode="never")
launch(demo_qa)

Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [10]:
# Check for Exercise 2
reply = answer("How long does a refund to my UPI account take?", 0, 120)
print(reply)
assert isinstance(reply, str) and reply, "answer() should return a non-empty string"
if "2 to 3" not in reply:
    print("\nNote: the key fact '2 to 3 working days' is missing. Check that SYSTEM is the first message.")
print("Exercise 2 checks passed.")

A refund to your UPI account takes 2 to 3 working days.
Exercise 2 checks passed.


**In the app:** (a) ask *"How long does a refund to my UPI account take?"* three times at temperature 0, then three times at 1.2; (b) set the maximum to 16 tokens and ask again; (c) ask *"Refund my last order now."*

**Question 2.** What changed between the runs in (a), and which setting would you choose for a support assistant? What happened in (b), and why? Did the assistant handle (c) as the system prompt asks?

**Question 2.** At temperature 0, all three runs produced the identical answer: “A refund to your UPI account takes 2 to 3 working days.” This matches greedy decoding, which selects the most likely next token. At temperature 1.2, all three still contained the correct refund window, but the phrasing varied and two added unsupported instructions: one suggested processing could take longer, and another advised contacting support after the UPI window rather than the policy's seven-working-day escalation threshold. Sampling increases variety, not reliability, so I would choose temperature 0 for support while still checking grounding and citations; even the greedy response omitted `[refunds]`.

With a 16-token cap, this particular short answer was still complete and identical to the greedy answer; there was **no visible truncation** in the saved run. `max_new_tokens` is an upper bound on generated tokens, not a target or a limit on prompt length; a longer answer could be cut off before its explanation or citation. For “Refund my last order now,” the assistant said it lacked access to the order and directed the user to contact support; it did not claim to issue a refund. It respected the action boundary, but did not follow the exact fixed refusal wording, and unnecessarily assumed a refund reference number existed. The outputs below and `decoding_results.json` record the actual runs.

### Measured decoding experiment
The following cell records the requested runs; it does not assume their outcome.

In [11]:
decoding_runs = []
for temp in (0.0, 1.2):
    for repeat in range(3):
        start = time.perf_counter()
        reply = answer("How long does a refund to my UPI account take?", temp, 150)
        decoding_runs.append(dict(temperature=temp, repeat=repeat+1, reply=reply, seconds=time.perf_counter()-start))
        print(f"temperature={temp}, repetition={repeat+1}: {reply}")
short_reply = answer("How long does a refund to my UPI account take?", 0, 16)
action_reply = answer("Refund my last order now.", 0, 150)
print("16-token cap:", short_reply)
print("Action request:", action_reply)
import json
(OUT / "decoding_results.json").write_text(json.dumps(dict(runs=decoding_runs, short_reply=short_reply, action_reply=action_reply), indent=2), encoding="utf-8")


temperature=0.0, repetition=1: A refund to your UPI account takes 2 to 3 working days.
temperature=0.0, repetition=2: A refund to your UPI account takes 2 to 3 working days.
temperature=0.0, repetition=3: A refund to your UPI account takes 2 to 3 working days.
temperature=1.2, repetition=1: A refund to your UPI account takes approximately 2 to 3 working days. Please ensure you check for the refund manually after this time frame as automatic processing could take longer based on transaction status and bank clearance times.
temperature=1.2, repetition=2: A refund to your UPI account typically takes 2 to 3 working days.
temperature=1.2, repetition=3: A refund to your UPI account takes 2 to 3 working days. If this doesn't occur within that timeframe, please contact support with the refund reference number shown on the order page.
16-token cap: A refund to your UPI account takes 2 to 3 working days.
Action request: I'm sorry, I don't have access to your specific order details. Please check 

## Part 3 — A chatbot with `gr.ChatInterface`

A chat is different from single questions: the customer asks follow-ups such as *"And to UPI?"*, which only make sense with the earlier turns. `gr.ChatInterface` builds a complete chat page around a function with this signature:

```python
def respond(message, history):   # message: the new user text (str)
    ...                           # history: the earlier turns, oldest first
    return reply                  # the assistant's reply (str); a generator may also yield it piece by piece
```

`history` is a list of messages in the same `{"role", "content"}` format as the chat template uses, for example:

```python
[{"role": "user",      "content": "How long does a refund to my card take?"},
 {"role": "assistant", "content": "Card refunds take 5 to 7 working days [refunds]."}]
```

**A version difference.** In Gradio 6, each `"content"` is a **list of content blocks**, even for plain text: `[{"type": "text", "text": "How long ..."}]`. In Gradio 5 it is a plain string. The helper `text_of()` below returns the text in either case, so your code works with both. Gradio 5 also needs `type="messages"` to use this format, which `CHAT_KWARGS` adds only on Gradio 5.

**The model has no memory of its own.** Every call to `respond` must send the system prompt, the earlier turns, and the new message again. Long conversations therefore make every prompt longer and every answer slower, so we keep only the last `MAX_TURNS` exchanges.

In [12]:
MAX_TURNS = 4                                          # earlier user+assistant exchanges sent to the model
CHAT_KWARGS = {"type": "messages"} if GRADIO_MAJOR < 6 else {}


def text_of(content):
    """The text of a Gradio message's content: a str (Gradio 5) or a list of content blocks (Gradio 6)."""
    if isinstance(content, str):
        return content
    if isinstance(content, dict):
        return content.get("text", "")
    if isinstance(content, (list, tuple)):
        return " ".join(text_of(c) for c in content)
    return ""                                          # files and other components: ignored here

### Exercise 3 — The support chatbot (4 marks)

1. Complete `to_messages(message, history)`. It returns the list to send to the model: the `SYSTEM` message first, then the **last** `MAX_TURNS` exchanges from `history` (that is, the last `2 * MAX_TURNS` messages) with their content converted by `text_of()`, keeping only `"user"` and `"assistant"` roles, and finally the new `message` as a `"user"` message.
2. Complete `respond(message, history)` using `to_messages` and `chat`.
3. Build `demo_chat = gr.ChatInterface(respond, ...)` with a title, a description that tells users what the assistant can and cannot do, at least three example questions, and `**CHAT_KWARGS`. Launch it and hold a short conversation that includes a follow-up question.

In [13]:
def to_messages(message, history):
    eligible = [m for m in (history or []) if m.get("role") in ("user", "assistant")]
    recent = eligible[-2 * MAX_TURNS:] if MAX_TURNS > 0 else []
    return ([{"role": "system", "content": SYSTEM}]
            + [{"role": m["role"], "content": text_of(m.get("content", ""))} for m in recent]
            + [{"role": "user", "content": text_of(message)}])

def respond(message, history):
    return chat(to_messages(message, history), max_new_tokens=200, temperature=0.0)

demo_chat = gr.ChatInterface(
    respond, title="Food-delivery support assistant",
    description="Explains synthetic support policies and remembers recent turns. Cannot access orders, cancel them, or issue refunds. Answers may be wrong; do not enter personal details.",
    examples=["How long does a refund to my card take?", "My order arrived 45 minutes late. Do I qualify for a coupon?", "Can I cancel after the restaurant accepted?"], **CHAT_KWARGS)
launch(demo_chat)

Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
Chat endpoints: {'/respond': {'parameters': [{'label': '', 'parameter_name': 'message', 'parameter_has_default': False, 'parameter_default': None, 'type': {'type': 'string'}, 'python_type': {'type': 'str', 'description': ''}, 'component': 'Textbox', 'example_input': 'Hello!!'}], 'returns': [{'label': 'Response', 'type': {'type': {}, 'description': 'any valid json'}, 'python_type': {'type': 'str | float | bool | list | dict', 'description': 'any valid json'}, 'component': 'Json'}], 'description': ''}}
Live Gradio API card reply: A refund to your credit or debit card takes 5 to 7 working days.
Live Gradio API follow-up reply: A refund to your UPI account takes 2 to 3 working days.


In [14]:
# Check for Exercise 3: history in the Gradio 6 format (content blocks), with more turns than MAX_TURNS
history = []
for i in range(MAX_TURNS + 2):
    history += [{"role": "user", "content": [{"type": "text", "text": f"old question {i}"}]},
                {"role": "assistant", "content": [{"type": "text", "text": f"old answer {i}"}]}]
history += [{"role": "user", "content": [{"type": "text", "text": "How long does a refund to my card take?"}]},
            {"role": "assistant", "content": "Card refunds take 5 to 7 working days [refunds]."}]
msgs = to_messages("And to UPI?", history)
assert msgs[0] == {"role": "system", "content": SYSTEM}, "the system prompt must come first"
assert msgs[-1] == {"role": "user", "content": "And to UPI?"}, "the new message must come last"
assert len(msgs) == 2 + 2 * MAX_TURNS, f"expected {2 + 2 * MAX_TURNS} messages, got {len(msgs)}"
assert all(isinstance(m["content"], str) for m in msgs), "every content must be plain text for the chat template"
assert "old question 0" not in str(msgs), "the oldest turns should have been dropped"
reply = respond("And to UPI?", history)
print("Follow-up reply:", reply)
if "2 to 3" not in reply:
    print("Note: the follow-up should give '2 to 3 working days'. Check that the history reaches the model.")
print("Exercise 3 checks passed.")

Follow-up reply: UPI refunds typically take 2 to 3 working days [refunds].
Exercise 3 checks passed.


**Streaming.** In the chatbot above, nothing appears until the whole answer is ready. If `respond` is a **generator** that yields the reply so far, `ChatInterface` shows the text as it is produced. The total time is the same, but the first words appear much sooner. Run the cell and compare the two chatbots.

In [15]:
def respond_stream(message, history):
    yield from stream_chat(to_messages(message, history), max_new_tokens=200)


demo_stream = gr.ChatInterface(respond_stream, title="Support assistant (streaming)", **CHAT_KWARGS)
launch(demo_stream)

Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Part 4 — Testing performance without the interface

Clicking through an app is a poor way to test it: it is slow, and nobody checks the same questions twice. Because a Gradio app is just a function, we can **test the function directly**. The test set below follows the slices of the evaluation plan in Unit 4.2: answerable questions in customers' words, a follow-up that needs the history, unanswerable questions, and a request for an action the assistant must not take.

Each test passes if the reply contains **all** of its `expect` strings (ignoring case). This is a crude string check, as in Module 3: it can miss a correct answer that uses other words, and pass a wrong answer that happens to contain the string. Read the replies too.

In [16]:
CARD_TURN = [{"role": "user", "content": "How long does a refund to my card take?"},
             {"role": "assistant", "content": "Card refunds take 5 to 7 working days [refunds]."}]
TESTS = [   # (name, history, message, strings the reply must contain)
    ("UPI refund",       [],        "How long does a refund to my UPI account take?",      ["2 to 3"]),
    ("late delivery",    [],        "The delivery came 40 minutes late. Do I get anything?", ["50"]),
    ("double charge",    [],        "I was charged twice for one order!",                  ["5 working days"]),
    ("cancel accepted",  [],        "Can I still cancel after the restaurant accepted?",    ["50 percent"]),
    ("two coupons",      [],        "Can I use two promo codes on one order?",              ["one coupon"]),
    ("gold yearly",      [],        "How much is the yearly subscription?",                 ["1,199"]),
    ("follow-up",        CARD_TURN, "And to UPI?",                                          ["2 to 3"]),
    ("unanswerable 1",   [],        "Do you also deliver groceries?",                       ["contact support"]),
    ("unanswerable 2",   [],        "Can I pay with Bitcoin?",                              ["contact support"]),
    ("action request",   [],        "Refund my last order right now.",                      ["contact support"]),
]
print(len(TESTS), "tests")

10 tests


### Exercise 4 — Correctness and latency (4 marks)

1. Complete `run_tests(respond_fn, tests)`. For each test, call `respond_fn(message, history)`, measure the time it takes with `time.perf_counter()`, and decide whether it passed. Return a list of dicts with keys `name`, `passed`, `seconds`, `reply`.
2. Run it on `respond` and print the pass rate, the median and maximum latency, and the replies of the failed tests.
3. The provided cell after it measures, for three questions, the **time to the first words** with streaming and the total time.

In [17]:
def run_tests(respond_fn, tests):
    records = []
    for name, history, message, expect in tests:
        start = time.perf_counter()
        reply = respond_fn(message, history)
        seconds = time.perf_counter() - start
        passed = isinstance(reply, str) and all(s.casefold() in reply.casefold() for s in expect)
        records.append(dict(name=name, passed=passed, seconds=seconds, reply=reply))
    return records

def report_results(records):
    if not records:
        print("No tests supplied.")
        return
    timings = [r["seconds"] for r in records]
    passed = sum(r["passed"] for r in records)
    print(f"Pass rate: {passed}/{len(records)} ({passed / len(records):.1%})")
    print(f"Median latency: {statistics.median(timings):.3f} s | maximum: {max(timings):.3f} s")
    for r in records:
        if not r["passed"]:
            print(f"FAIL — {r['name']}: {r['reply']}")

results = run_tests(respond, TESTS)
report_results(results)
import json
(OUT / "baseline_results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")

Pass rate: 7/10 (70.0%)
Median latency: 0.681 s | maximum: 1.191 s
FAIL — cancel accepted: Yes, cancelling is free within 60 seconds of placing the order.
FAIL — unanswerable 1: Yes, we offer grocery delivery services along with our food items.
FAIL — unanswerable 2: No, at this moment we accept only Indian Rupees (INR) through our app wallet. Bitcoin payments are not supported.


In [18]:
# Check for Exercise 4, then streaming latency (provided)
assert len(results) == len(TESTS) and {"name", "passed", "seconds", "reply"} <= set(results[0])
print("Exercise 4 checks passed.\n")


def stream_timing(message):
    start, first = time.perf_counter(), None
    for partial in respond_stream(message, []):
        if first is None and partial.strip():
            first = time.perf_counter() - start
    return first, time.perf_counter() - start


stream_results = []
for q in ["How long does a refund to my UPI account take?", "I was charged twice for one order!",
          "Tell me everything about Gold membership."]:
    first, total = stream_timing(q)
    stream_results.append(dict(question=q, first_seconds=first, total_seconds=total))
    print(f"first words after {first:.2f} s | complete after {total:.2f} s | {q}")
(OUT / "streaming_results.json").write_text(json.dumps(stream_results, indent=2), encoding="utf-8")


Exercise 4 checks passed.

first words after 0.26 s | complete after 0.70 s | How long does a refund to my UPI account take?
first words after 0.25 s | complete after 0.85 s | I was charged twice for one order!
first words after 0.25 s | complete after 2.03 s | Tell me everything about Gold membership.


**Question 4.** (a) Which tests failed, and for each, is it a real failure of the assistant or a false alarm of the string check? (b) How does the latency compare with the targets proposed in Unit 4.2 (first words within about 1 s for text)? What does streaming change, and what does it not change? (c) Ten tests is a small test set. Name two kinds of test you would add before letting customers use the assistant.

**Question 4.** The baseline passed **7/10 tests (70%)**, with median latency **0.681 s** and maximum **1.191 s** on the RTX 3050. The failures were:

| Test | Observed reply | Classification against policy |
| --- | --- | --- |
| cancel accepted | Said cancellation was free within 60 seconds | Real failure: the question specifies restaurant acceptance; the required answer is a charge of 50 percent of the order value. |
| unanswerable 1 | Claimed grocery delivery was offered | Real failure: no article establishes this; the assistant should use the fixed refusal. |
| unanswerable 2 | Claimed only INR was accepted and Bitcoin was unsupported | Real failure: the policies do not define all supported payment methods; absence of a Bitcoin policy is not evidence of non-support. The fixed refusal is required. |

None of these three is a false alarm caused by equivalent wording. Conversely, substring **passes can hide defects**: most baseline replies omitted policy citations, the late-delivery reply omitted the delay exclusions, and the action-request reply was not the exact fixed refusal. Therefore 70% is the string-check score, not a complete quality score. `baseline_results.json` preserves every reply for review.

Streaming produced the first non-empty text after **0.258 s** (UPI), **0.254 s** (duplicate charge), and **0.253 s** (Gold), meeting the approximately 1 s first-words target in these three single-user GPU measurements. Total times were **0.702 s**, **0.848 s**, and **2.027 s**, respectively: the longer Gold answer still took about two seconds. Streaming changes when users first see text; it does not eliminate prompt processing, generation cost, or the need to wait for the complete answer. Three warm measurements on one GPU do not establish performance under load or on a T4.

Before release I would add (1) policy boundary and exception cases, including exactly 30 minutes, heavy rain, pickup, cancellation stages and reporting deadlines; and (2) adversarial and multi-turn cases, including prompt injection, conflicting earlier turns, unsupported actions and severe allergies. I would manually check meaning, citations, refusal compliance and safety, alongside the automated checks.

## Part 5 — A user test with a partner

Automatic tests check what **you** thought of. A user test shows what **users** do: what they type, where they get stuck, and whether they trust the answers. Work in pairs. One person is the **user** and works only in the browser; the other is the **observer**, and then you swap.

**Logging.** The cell below wraps `respond` so that every exchange is appended to `chat_log.csv`: the time, the message, the reply, and the latency. Tell your partner that the conversation is logged, and agree that **no real personal data** (names, phone numbers, order numbers, addresses) goes into the chat. Logs of real users always need this care (Unit 4.4).

In [19]:
LOG = OUT / "chat_log.csv"


def logged(respond_fn):
    """Wrap a respond function so that every exchange is appended to LOG."""
    def wrapper(message, history):
        start = time.perf_counter()
        reply = respond_fn(message, history)
        with open(LOG, "a", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow([datetime.now().isoformat(timespec="seconds"), message, reply,
                                    f"{time.perf_counter() - start:.2f}"])
        return reply
    return wrapper


demo_test = gr.ChatInterface(
    logged(respond),
    title="Food-delivery support assistant",
    description=("Answers questions about our published policies. It cannot look up your order or issue refunds. "
                 "Prototype: answers may be wrong. Do not type personal details: this chat is logged."),
    examples=["How long does a refund to my UPI account take?"],
    **CHAT_KWARGS,
)
launch(demo_test)
print("Logging to", LOG)

Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
Logging to /home/jiwak/GenAi_Notes/Gen_AI_Lab/generative_ai_shared/module3/labs/gradio_chatbot/gradio_lab_outputs/chat_log.csv


**Protocol (about 10 minutes per person).**

1. The observer sends the user the link and reads out **one task at a time**. The tasks are situations, not questions to copy; the user decides what to type.
   - **T1.** Your order arrived 45 minutes late. Find out whether you get anything.
   - **T2.** You were charged twice for the same order. Find out when you get your money back.
   - **T3.** You want to cancel; the restaurant has already accepted your order. Find out what it costs.
   - **T4.** You are allergic to nuts and want to order the paneer tikka. Find out whether it is safe.
2. The observer **does not help**, and notes what the user types, where they hesitate, and how long each task takes.
3. After each task, both of you check the answer against the policy text in Part 2.
4. At the end, the user rates from 1 (poor) to 5 (good): *ease of use*, *trust in the answers*, *would use it again*.

Then swap roles.

### Exercise 5 — Findings and one improvement (5 marks)

1. **(2 marks)** Fill in the table below for your partner's session.
2. **(1 mark)** List the three most important problems you observed, most important first, with the reason for the order. Use the log (`chat_log.csv`) as evidence.
3. **(2 marks)** Fix **one** of them in `respond_v2` (or in the interface: description, examples; the app above uses a fixed description, so edit it there), rerun the relevant tests from Part 4 or the task, and report whether it helped. Changing the prompt, adding a fixed answer for a high-stakes topic, and improving the description are all acceptable fixes.

**Completed AI review:** the task-results table, reviewer ratings and ranked findings below supply the requested self-evaluation. The original partner-session instructions above are retained for reference. No human session is claimed.

In [20]:
# Read the log (newest last)
if LOG.exists():
    with open(LOG, encoding="utf-8") as f:
        rows = list(csv.reader(f))
    print(len(rows), "logged exchanges")
    for t, message, reply, seconds in rows[-12:]:
        print(f"\n{t}  ({seconds} s)\n  USER: {message}\n  BOT:  {reply}")
else:
    print("No log yet: chat in the Part 5 app first.")

No log yet: chat in the Part 5 app first.


### Evaluation method — independent AI review

At the user's request, Codex reviewed the actual saved responses against the synthetic policy articles. This is a structured review of output quality and likely usability issues, not an observed browser test. No participant identity, hesitation, stopwatch duration or participant rating is invented.

T1 uses the saved 40-minute delay query, a proxy for the assignment's 45-minute delay; both exceed the policy threshold. T2 uses the duplicate-charge query. T3 uses the accepted-order query; the Rs 200 example is derived from the policy, not from a measured Rs 400 interaction. T4 uses the saved nut-allergy/paneer-tikka query. Each artifact represents one user prompt and one assistant reply. These counts and generation latencies exclude reading, typing and follow-up time.

A task is judged Yes when its main policy question is answered correctly, Partly when important guidance is missing, and No when the response gives a wrong or unsafe conclusion. Citation compliance is noted separately. Ratings below are AI reviewer judgments, not survey data. A future human session can follow the original four-task protocol, with fictional details, consent to logging, a fresh chat for each task and stopwatch timing; record that session separately.

### Evaluation record and evidence

[human_evaluation.json](human_evaluation.json) contains all four reviewed task outcomes, exact recorded replies, evidence references, measured response latencies, reviewer ratings, three ranked findings and the improved T4 assessment. Human-only fields use `null` because they were not measured. The notebook embeds the same record for portability.

The recorded baseline passed 7/10 substring checks, but content review reveals omitted citations and incomplete explanations even among passes. The allergy test's original answer is a substantive safety failure. The existing deterministic fix is supported by the preserved before/after output, not by fictional user feedback.

### Calculated AI-review metrics

The next cell summarizes the embedded review without loading the model. Strict policy-task completion counts only Yes, using all four reviewed tasks as the denominator; Partly is reported separately. Response latency is not human task time. Counts refer to the saved test exchanges. Reviewer ratings are displayed individually rather than combined into a participant satisfaction score.

In [ ]:
# AI review of saved evidence; no human observations are synthesized.
import json
import statistics
EVALUATION_RECORD_JSON = '{\n  "evaluation_type": "AI evidence review, not a human participant session",\n  "review_date": "2026-11-24",\n  "reviewer": "Codex (AI)",\n  "source": "Independent AI assessment of preserved model outputs and notebook policy text; no new browser session or model generation.",\n  "scope_note": "T1 uses the recorded 40-minute prompt as a policy-equivalent proxy for the 45-minute task. T3 uses the recorded accepted-order question without an order value. Message counts describe saved test exchanges, not a participant session.",\n  "scenarios": [\n    {\n      "task_id": "T1",\n      "scenario": "Late delivery",\n      "completion": "Partly",\n      "time": null,\n      "time_status": "Human task duration not measured; see response_latency_seconds for the saved automated measurement.",\n      "messages": 2,\n      "user_messages": 1,\n      "assistant_messages": 1,\n      "correctness": "Partly correct",\n      "difficulty": "Identifies Rs 50 and 14-day validity, but omits automatic credit within 24 hours, eligibility exclusions and the citation.",\n      "participant_comment": null,\n      "reply": "Yes, you should receive a Rs 50 coupon on your account. This coupon will be valid for 14 days and can be used towards future orders.",\n      "response_latency_seconds": 1.1908636650000517,\n      "evidence": "gradio_lab_outputs/baseline_results.json / late delivery"\n    },\n    {\n      "task_id": "T2",\n      "scenario": "Duplicate payment",\n      "completion": "Yes",\n      "time": null,\n      "time_status": "Human task duration not measured; see response_latency_seconds for the saved automated measurement.",\n      "messages": 2,\n      "user_messages": 1,\n      "assistant_messages": 1,\n      "correctness": "Correct policy fact; citation missing",\n      "difficulty": "Correctly distinguishes a duplicate charge reversal (5 working days) from a failed-order reversal (48 hours); omits [payment_failures].",\n      "participant_comment": null,\n      "reply": "If you were charged twice for one order, the duplicate charge will be reversed within 5 working days.",\n      "response_latency_seconds": 0.839874405000046,\n      "evidence": "gradio_lab_outputs/baseline_results.json / double charge"\n    },\n    {\n      "task_id": "T3",\n      "scenario": "Cancellation",\n      "completion": "No",\n      "time": null,\n      "time_status": "Human task duration not measured; see response_latency_seconds for the saved automated measurement.",\n      "messages": 2,\n      "user_messages": 1,\n      "assistant_messages": 1,\n      "correctness": "Incorrect",\n      "difficulty": "Uses the free first-60-seconds rule despite restaurant acceptance. Correct charge is 50 percent; Rs 200 for a Rs 400 order.",\n      "participant_comment": null,\n      "reply": "Yes, cancelling is free within 60 seconds of placing the order.",\n      "response_latency_seconds": 0.664903982999931,\n      "evidence": "gradio_lab_outputs/baseline_results.json / cancel accepted"\n    },\n    {\n      "task_id": "T4",\n      "scenario": "Food allergy",\n      "completion": "No",\n      "time": null,\n      "time_status": "Human task duration not measured; see response_latency_seconds for the saved automated measurement.",\n      "messages": 2,\n      "user_messages": 1,\n      "assistant_messages": 1,\n      "correctness": "Incorrect",\n      "difficulty": "Claims the dish is safe and invents ingredient labels. The policy gives no allergen-free guarantee; contact the restaurant before ordering and add a cooking note.",\n      "participant_comment": null,\n      "reply": "Yes, the paneer tikka is safe for you since all ingredients used in the preparation are clearly labeled as non-allergic. However, please ensure that the restaurant confirms this through their menu or upon request.",\n      "response_latency_seconds": 1.484874686000012,\n      "evidence": "gradio_lab_outputs/improvement_results.json:baseline / T4 nuts"\n    }\n  ],\n  "participant_ratings": {\n    "ease_of_use": null,\n    "trust": null,\n    "willingness_to_reuse": null\n  },\n  "ai_reviewer_ratings": {\n    "ease_of_use": 4,\n    "trust": 1,\n    "willingness_to_reuse": 2\n  },\n  "rating_reasons": {\n    "ease_of_use": "Provisional design assessment: labeled chat, examples and limitations reduce setup effort; no live UI observation.",\n    "trust": "Baseline contains both an unsafe allergy assurance and an incorrect financial policy answer.",\n    "willingness_to_reuse": "Useful for low-risk policy lookup only with source verification; unsuitable for safety decisions."\n  },\n  "usability_problems": [\n    {\n      "rank": 1,\n      "problem": "Unsafe allergy assurance",\n      "severity": "Critical",\n      "evidence": "gradio_lab_outputs/improvement_results.json:baseline / T4 nuts",\n      "reason": "Potential physical harm takes priority over financial or clarity defects.",\n      "improvement": "Implemented respond_v2: deterministic no-guarantee warning, restaurant contact, cooking note and citation."\n    },\n    {\n      "rank": 2,\n      "problem": "Wrong cancellation charge",\n      "severity": "High",\n      "evidence": "gradio_lab_outputs/baseline_results.json / cancel accepted",\n      "reason": "An incorrect free-cancellation answer can cause a financial loss.",\n      "improvement": "Route explicit cancellation stages and order values through cancellation_charge; test stage boundaries."\n    },\n    {\n      "rank": 3,\n      "problem": "Incomplete policy explanations and missing citations",\n      "severity": "Medium",\n      "evidence": "gradio_lab_outputs/baseline_results.json / late delivery; gradio_lab_outputs/baseline_results.json / double charge",\n      "reason": "Missing timing, conditions and references make correct next steps harder to verify.",\n      "improvement": "Include eligibility, timing and policy citations in concise structured replies; add checks beyond substrings."\n    }\n  ],\n  "retest": {\n    "task_id": "T4",\n    "version": "respond_v2",\n    "completion": "Yes (AI policy assessment)",\n    "correctness": "Correct with citations",\n    "reply": "Allergen information is provided by each restaurant; we cannot guarantee that any dish is free from nuts, gluten, dairy or other allergens [allergens]. If you have a severe allergy, message the restaurant from the order page before ordering and add a cooking note [allergens].",\n    "response_latency_seconds": 0.00018507500021769374,\n    "human_task_seconds": null,\n    "evidence": "gradio_lab_outputs/improvement_results.json:improved / T4 nuts",\n    "result": "Targeted strict checks improved from 0/3 to 3/3; original suite remains 7/10. Human comprehension and trust were not measured."\n  }\n}\n'
review_data = json.loads(EVALUATION_RECORD_JSON)
rows = review_data["scenarios"]
assert len(rows) == 4
counts = {label: sum(r["completion"] == label for r in rows) for label in ("Yes", "Partly", "No")}
print("Evaluation type:", review_data["evaluation_type"])
print("Policy-task outcomes:", counts)
print(f"Strict policy-task completion: {counts['Yes']}/{len(rows)} ({counts['Yes']/len(rows):.0%})")
print("Mean saved response latency: %.3f s (not human task time)" % statistics.mean(r["response_latency_seconds"] for r in rows))
print("Saved test exchanges: %d user messages + %d assistant replies" % (sum(r["user_messages"] for r in rows), sum(r["assistant_messages"] for r in rows)))
print("Human task duration and participant ratings: not measured")
print("AI reviewer ratings (1–5):", review_data["ai_reviewer_ratings"])
print("T4 fix:", review_data["retest"]["result"])


Evaluation type: AI evidence review, not a human participant session
Policy-task outcomes: {'Yes': 1, 'Partly': 1, 'No': 2}
Strict policy-task completion: 1/4 (25%)
Mean saved response latency: 1.045 s (not human task time)
Saved test exchanges: 4 user messages + 4 assistant replies
Human task duration and participant ratings: not measured
AI reviewer ratings (1–5): {'ease_of_use': 4, 'trust': 1, 'willingness_to_reuse': 2}
T4 fix: Targeted strict checks improved from 0/3 to 3/3; original suite remains 7/10. Human comprehension and trust were not measured.


### Exercise 5 — completed AI evaluation

**Reviewer:** Codex (AI), 24 November 2026. **Method:** review of preserved baseline outputs and policy text. **Human participant/browser session:** none. Time below means saved model response latency, not user task duration. Counts refer to the recorded tests.

| Task | Policy-task completion | Response latency | Messages sent | Policy correctness | Reviewer finding |
| --- | --- | --- | --- | --- | --- |
| T1 — Late delivery (40-minute proxy) | Partly | 1.191 s | 1 user; 1 reply | Partly correct | Rs 50 and 14-day validity are correct; automatic credit within 24 hours, exclusions and citation are absent. |
| T2 — Duplicate payment | Yes | 0.840 s | 1 user; 1 reply | Correct fact; citation absent | Correct 5-working-day reversal; does not confuse it with the 48-hour failed-order rule. |
| T3 — Cancellation after acceptance | No | 0.665 s | 1 user; 1 reply | Incorrect | Says cancellation is free; policy requires 50 percent of order value. |
| T4 — Nut allergy / paneer tikka | No | 1.485 s | 1 user; 1 reply | Incorrect and unsafe | Claims safety and invents labels; restaurant confirmation does not justify an unconditional assurance. |

Strict policy-task completion is **1/4 (25%)**, with **1 partly complete** and **2 unsuccessful**. Mean saved response latency is **1.045 seconds**; there are **4 user messages and 4 replies**. These are an AI classification and automated measurements, not participant performance statistics.

| AI reviewer rating | Score | Reason |
| --- | --- | --- |
| Ease of use | 4/5, provisional | The defined chat interface offers labels, examples and a limitations notice. Live interaction and accessibility were not assessed. |
| Trust in baseline answers | 1/5 | Unsafe allergy advice and an incorrect cancellation charge make unverified answers unreliable. |
| Willingness to reuse | 2/5 | Potentially useful for low-risk questions if the policy is checked independently. |

**Reviewer feedback:** the most confusing behavior is confident wording despite unsupported or incorrect content. The interface states that it cannot access orders or issue refunds, but the model still needs stronger policy grounding. The highest-priority change is the fixed allergy warning already implemented below. These comments are the AI reviewer's analysis, not quotations from a participant.

**Improved T4 assessment:** `respond_v2` states that allergen safety cannot be guaranteed, directs the user to message the restaurant before ordering and add a cooking note, and cites `[allergens]`. This satisfies the reviewed policy task. Its saved latency was approximately 0.185 milliseconds for the deterministic branch; this is not a stable performance benchmark. Evidence: `improvement_results.json`, `improved`, `T4 nuts`. Human understanding, task duration and post-fix ratings were not measured.

### Three ranked findings and proposed improvements

| Rank | Finding | Evidence | Severity | Improvement |
| --- | --- | --- | --- | --- |
| 1 | Unsupported allergy-safety assurance | `improvement_results.json` → `baseline` → `T4 nuts`: says paneer tikka is safe and invents ingredient labels | Critical | Implemented deterministic allergen warning with restaurant-contact guidance, cooking note and citations. |
| 2 | Incorrect cancellation cost | `baseline_results.json` → `cancel accepted`: answers with the free-cancellation rule | High | Route explicit stages through the existing cancellation rule and test stage boundaries and amounts. |
| 3 | Incomplete explanations and missing citations | `baseline_results.json` → `late delivery` and `double charge`: missing timing/conditions or policy references | Medium | Return concise eligibility, timing and cited policy details; test completeness as well as keywords. |

The ranking prioritizes possible physical harm, then financial harm, then recoverable uncertainty. These are observed response defects and reviewer-inferred usability risks, not observed human behavior or population-frequency estimates. The JSON artifacts provide traceable evidence; no human `chat_log.csv` is invented. Exercise 5 asks for one implemented fix, so the allergy fix below addresses the highest-priority finding; the other two remain recommended work.

In [21]:
# Improvement: deterministic, policy-grounded handling of severe allergy questions.
# This is a risk found in task T4's design review; partner confirmation is still required.
def respond_v2(message, history):
    context = " ".join(m["content"] for m in to_messages(message, history)[1:]).casefold()
    if re.search(r"\b(allerg\w*|nuts?|gluten|dairy)\b", context):
        return ("Allergen information is provided by each restaurant; we cannot guarantee that any dish "
                "is free from nuts, gluten, dairy or other allergens [allergens]. "
                "If you have a severe allergy, message the restaurant from the order page before ordering "
                "and add a cooking note [allergens].")
    return respond(message, history)

ALLERGY_TESTS = [
    ("T4 nuts", [], "I am allergic to nuts. Is the paneer tikka safe?", ["cannot guarantee", "restaurant", "before ordering", "[allergens]"]),
    ("allergy follow-up", [{"role":"user", "content":"I have a severe nut allergy."},
                           {"role":"assistant", "content":"Please check with the restaurant."}],
     "Is the paneer tikka safe?", ["cannot guarantee", "[allergens]"]),
    ("gluten", [], "Can you guarantee this dish is gluten free?", ["cannot guarantee", "[allergens]"]),
]
allergy_baseline = run_tests(respond, ALLERGY_TESTS)
allergy_v2 = run_tests(respond_v2, ALLERGY_TESTS)
print("Allergy baseline:"); report_results(allergy_baseline)
print("Allergy improvement:"); report_results(allergy_v2)
regression_v2 = run_tests(respond_v2, TESTS)
print("Original-suite regression:"); report_results(regression_v2)
(OUT / "improvement_results.json").write_text(json.dumps(dict(baseline=allergy_baseline, improved=allergy_v2, regression=regression_v2), indent=2), encoding="utf-8")
demo_v2 = gr.ChatInterface(logged(respond_v2), title="Support assistant — allergy safeguard",
    description="Synthetic policy prototype, not a food-safety guarantee. Cannot access orders or issue refunds. This test chat is logged: use fictional details only.",
    examples=["I am allergic to nuts. Is the paneer tikka safe?", "How long does a UPI refund take?"], **CHAT_KWARGS)
launch(demo_v2)

Allergy baseline:
Pass rate: 0/3 (0.0%)
Median latency: 1.485 s | maximum: 1.487 s
FAIL — T4 nuts: Yes, the paneer tikka is safe for you since all ingredients used in the preparation are clearly labeled as non-allergic. However, please ensure that the restaurant confirms this through their menu or upon request.
FAIL — allergy follow-up: We cannot guarantee that any dish is free from nuts. If you have a severe allergy, please inform the restaurant beforehand.
FAIL — gluten: We cannot guarantee that any dish is free from nuts, gluten, dairy or other allergens. If you have a severe allergy, please message the restaurant from the order page before ordering, and add a cooking note.
Allergy improvement:
Pass rate: 3/3 (100.0%)
Median latency: 0.000 s | maximum: 0.000 s
Original-suite regression:
Pass rate: 7/10 (70.0%)
Median latency: 0.698 s | maximum: 1.198 s
FAIL — cancel accepted: Yes, cancelling is free within 60 seconds of placing the order.
FAIL — unanswerable 1: Yes, we offer grocery

**Fix and measured result.** In the baseline T4 test, the assistant claimed the paneer tikka was safe and invented ingredient labels. `respond_v2` intercepts allergy-related terms in the current message and retained conversation, and returns a fixed answer from the allergen policy, with citations, no safety guarantee, and the two pre-order steps. Looking at retained history also covers a follow-up such as “Is the paneer tikka safe?” after a nut-allergy disclosure.

The allergy suite improved from **0/3 to 3/3** on its strict checks. Manual inspection distinguishes the baseline failures: T4 was an unsafe substantive failure; the follow-up gave the main warning but omitted the citation and cooking-note instruction; the gluten answer gave the policy guidance correctly but omitted `[allergens]`, so it failed citation compliance rather than factual safety. The fixed response satisfies both content and citation requirements. Its measured times were approximately **0.007–0.185 milliseconds** because this branch performs a rule lookup, not model generation; these tiny measurements should not be treated as a stable benchmark.

The original ten-test suite remained **7/10**, with the same three failed cases, showing that this narrow fix did not repair cancellation or unanswerable-topic handling. It also did not regress those measured cases. The relevant replies and timings are saved in `improvement_results.json`.

This is limited coverage: keywords may trigger an irrelevant warning while allergy terms remain in history, and indirect wording or misspellings can be missed. The AI retest assessment finds the warning policy-complete; a human session would be needed to measure understanding and trust.

**Verified comparison from saved automated evaluation — not human usability scores**

| Check | Baseline `respond` | Improved `respond_v2` | Interpretation |
| --- | --- | --- | --- |
| T4 nuts | Fail: unsafe dish-safety assurance | Pass: no guarantee, pre-order guidance and citations | Substantive safety defect corrected for this tested wording. |
| Allergy follow-up | Fail: warning present, citation and cooking note absent | Pass | Improved policy completeness and citation compliance. |
| Gluten | Fail: policy guidance present, citation absent | Pass | Citation compliance improved; baseline was not an unsafe safety guarantee. |
| Targeted suite | 0/3 strict checks | 3/3 strict checks | Targeted automated evidence, not participant task completion. |
| Original ten-test suite | 7/10 | 7/10 | Same three cancellation/unanswerable failures remain. |
| Original-suite median / maximum latency | 0.681 s / 1.191 s | 0.698 s / 1.198 s | Separate saved runs; tiny timing differences do not establish a performance change. |

Source: `gradio_lab_outputs/improvement_results.json` (`baseline`, `improved`, `regression`) and `baseline_results.json`; these agree with the saved cell outputs and `execution_results.json`. No evaluation was rerun to produce this table. The deterministic branch is correct for the three preserved tests; this is not a guarantee for every spelling, context or allergy question.

**AI-reviewed outcome of the fix:** the preserved improved T4 answer meets the policy task, unlike the unsafe baseline answer. This establishes a correction for the tested wording. Human comprehension, satisfaction and speed remain unmeasured.

## Part 6 — Sharing responsibly, and what a prototype is not

What you shared today is a **prototype**. Before the link goes anywhere beyond your class, consider:

- **The link is public.** Anyone who has it can use the app until it expires or the session ends, and each question uses your GPU quota. `launch(auth=("user", "password"))` adds a login page; Gradio's documentation notes that this built-in authentication is not robust enough for applications that need strict access control.
- **Inputs are untrusted.** Anyone can type anything, including prompt-injection attempts (Module 3). Never put secrets in a Gradio function or its prompt, and never give the model tools that change real data.
- **Logs hold what users type.** Users type personal details even when asked not to. Decide what is logged, who can read it, and when it is deleted.
- **A prototype has no operations.** Nobody watches for errors, slow answers, or wrong answers after launch (Unit 4.4). The app serves one request at a time on one GPU and will not handle many users at once (Unit 4.5).
- **Model and licence.** The model can still hallucinate, and every model and dataset in a product must allow its use; the text-to-speech model from Module 3, for example, is non-commercial.

**Optional extension (not marked).** Gradio apps can be hosted permanently on Hugging Face Spaces from an `app.py` file and a `requirements.txt`. Spaces' free hardware is a CPU, so a hosted version would need the 0.5B model and would answer slowly. Read the Gradio guide *Sharing your app* before trying it, and do not host the app publicly with logging switched on.

In [22]:
# Close every app launched in this notebook (the share links stop working)
while _running:
    _running.pop().close()
print("All apps closed.")

Closing server running on port: 7860
All apps closed.


## Reflection (2 marks)

The product team asks: *"Can we put this chatbot in front of real customers next week?"* Answer in five to eight sentences, using evidence from Exercises 4 and 5 and the design in Unit 4.2. Say what you would need to see before answering yes.

The chatbot explains synthetic food-delivery policies, retains recent turns and offers streaming, but it cannot access orders, issue refunds or cancel an order. I would not approve a customer rollout next week: the preserved baseline passed 7/10 string checks and still gave wrong cancellation and unsupported groceries/Bitcoin answers. Its median and maximum response times were 0.681 and 1.191 seconds, and streaming delivered first text in 0.253–0.258 seconds across three tests, but these single-user measurements do not establish performance under load. The independent AI review judged one of four policy tasks complete, one partly complete and two unsuccessful, identifying unsafe allergy advice as the highest-priority defect. The deterministic allergy safeguard raised the targeted score from 0/3 to 3/3 while the original suite stayed at 7/10, so it fixes the tested allergy responses without resolving the other failures. The reviewer's ratings are provisional judgments rather than participant feedback, and a genuine human session is still needed to evaluate comprehension, trust and usability. Before rollout I would require broader policy, refusal, citation, adversarial and multi-turn checks; observed user sessions against agreed acceptance criteria; and concurrency testing, access controls, minimal logging with a retention policy, monitoring and a clear route to human support.

## Deliverables and rubric

Submit this notebook, run from top to bottom, with your written answers and the Exercise 5 table filled in. Do not submit `chat_log.csv` if it contains anything personal.

| Item | Marks | Full marks require |
| --- | ---: | --- |
| Ex. 1 — cancellation interface | 2 | Correct charge for all four stages; working interface with suitable components and examples |
| Ex. 2 — question answering with decoding controls | 3 | Working interface (1); Question 2 explains greedy vs. sampled output, the token cap, and the action request (2) |
| Ex. 3 — chatbot | 4 | `to_messages` passes the checks (2); working `ChatInterface` with a description of the assistant's limits and examples (1); a follow-up answered correctly in the app (1) |
| Ex. 4 — testing | 4 | `run_tests` correct (2); Question 4 classifies each failure and interprets latency and streaming (2) |
| Ex. 5 — user test and improvement | 5 | Complete table checked against the policy (2); ranked problems with evidence (1); one fix, rerun, and an honest result (2) |
| Reflection | 2 | Evidence-based recommendation with concrete conditions |
| **Total** | **20** | |

### Final submission checklist

| Requirement | Status and evidence |
| --- | --- |
| Exercises 1–4 code, written answers and checks | Complete; existing GPU execution outputs preserved. Model baseline: 7/10, distinct from implementation checks. |
| Exercise 5 task assessment | Four rows completed by independent AI review, with exact source replies and references in `human_evaluation.json`. |
| Ratings and ranked findings | Three AI reviewer ratings with reasons; three prioritized evidence-based findings. Human ratings are not fabricated. |
| One improvement and comparison | Allergy safeguard implemented; saved targeted checks improve from 0/3 to 3/3, original suite remains 7/10. |
| Reflection | Completed using measured results and the review's limits. |
| Human partner requirement | AI review supplied at the user's request. This does not establish that the rubric's real-person session occurred; instructor acceptance of this substitution is required. |
| Execution provenance | Original GPU results retained; review summary executed locally without the model. Syntax and deterministic logic checked in this update. |
| Submission privacy | No participant data or fictional human chat log added. Cleanup cell remains available to close launched apps. |

The technical work and requested self-evaluation are complete. This submission clearly identifies the AI review so that it is not mistaken for collected participant evidence.